# Aula 7 — Seleção de modelos e tuning

**Como comparar configurações sem enganar a avaliação**

Na Aula 6, aprendemos que avaliar bem exige olhar além da acurácia.

Agora surge uma nova pergunta:

> Se podemos testar diferentes configurações, como escolher a melhor sem usar o conjunto de teste como parte do processo de decisão?

Nesta aula vamos introduzir **validação cruzada** e `GridSearchCV` para comparar configurações de um pipeline `TF-IDF + MultinomialNB` de forma mais disciplinada.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar por que ajustar hiperparâmetros olhando o teste gera risco de overfitting;
- explicar o papel da validação cruzada;
- distinguir parâmetro de hiperparâmetro;
- usar `GridSearchCV` em um pipeline de texto;
- comparar configurações de `TfidfVectorizer` e `MultinomialNB`;
- interpretar `best_params_`, `best_score_` e resultados de validação;
- preservar um conjunto de teste final para avaliação honesta.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.


## 2. O perigo de otimizar olhando o teste

Imagine que você teste várias configurações e escolha aquela que obteve a melhor nota no conjunto de teste.

O problema é que, ao fazer isso repetidamente, o teste deixa de ser uma avaliação independente e passa a influenciar suas decisões.

Isso pode gerar uma forma de **overfitting ao conjunto de teste**.

Por isso, o teste final deve ser preservado para o fim do processo.


## 3. Validação cruzada

Na validação cruzada, o conjunto de treinamento é dividido em partes menores chamadas *folds*.

O modelo é treinado e validado várias vezes, alternando qual parte funciona como validação.

A ideia simplificada é:

```text
treino
→ fold 1 valida
→ fold 2 valida
→ fold 3 valida
→ média dos resultados
```

Isso reduz a dependência de uma única divisão de dados.


## 4. Parâmetros e hiperparâmetros

**Parâmetros** são aprendidos pelo modelo durante o treinamento.

**Hiperparâmetros** são configurações definidas antes do treinamento e que controlam o comportamento do algoritmo.

Exemplos nesta aula:

- `ngram_range` do `TfidfVectorizer`;
- `min_df` do `TfidfVectorizer`;
- `alpha` do `MultinomialNB`.


## 5. Dataset de exemplo

Vamos trabalhar com um pequeno conjunto balanceado de mensagens rotuladas.


In [ ]:
texts = [
    "como altero minha senha",
    "onde vejo minha fatura",
    "posso pagar amanhã",
    "como atualizo meu cadastro",
    "qual o prazo para resposta",
    "como cancelo o serviço",
    "meu pedido não chegou",
    "o atendimento foi péssimo",
    "estou insatisfeito com o serviço",
    "a entrega atrasou novamente",
    "o suporte não resolveu meu problema",
    "estou muito irritado com o atendimento",
    "o atendimento foi excelente",
    "fui muito bem atendido",
    "serviço rápido e eficiente",
    "estou satisfeito com o atendimento",
    "a equipe resolveu tudo rapidamente",
    "gostei muito do suporte",
]

labels = [
    "duvida", "duvida", "duvida", "duvida", "duvida", "duvida",
    "reclamacao", "reclamacao", "reclamacao", "reclamacao", "reclamacao", "reclamacao",
    "elogio", "elogio", "elogio", "elogio", "elogio", "elogio",
]


## 6. Preserve o teste final

Primeiro separamos uma parte dos dados que não será usada no tuning.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.33,
    random_state=42,
    stratify=labels,
)

print("Treino para seleção/tuning:", len(X_train))
print("Teste final preservado:", len(X_test))


## 7. Pipeline base

Vamos reaproveitar a arquitetura da Aula 5.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB()),
])


## 8. Definindo o espaço de busca

Agora escolhemos algumas configurações candidatas.

Mantemos a grade pequena para que o experimento continue didático e rápido.


In [ ]:
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__alpha": [0.5, 1.0],
}

param_grid


### Como ler os nomes

Em um `Pipeline`, usamos o padrão:

```text
nome_da_etapa__nome_do_hiperparametro
```

Por isso temos, por exemplo, `tfidf__ngram_range` e `classifier__alpha`.


## 9. Executando o GridSearchCV

Vamos usar F1 macro como métrica de seleção, porque queremos considerar as três classes com o mesmo peso.


In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Melhor F1 macro de validação:", round(search.best_score_, 3))
print("Melhores hiperparâmetros:", search.best_params_)


### O que interpretar

`best_score_` representa o desempenho médio de validação cruzada da melhor configuração encontrada.

`best_params_` mostra a combinação escolhida pelo processo de busca.

Isso ainda **não é** o resultado final no conjunto de teste.


## 10. Inspecionando as configurações testadas

Os resultados completos ficam em `cv_results_`.


In [ ]:
import pandas as pd

results = pd.DataFrame(search.cv_results_)

results[[
    "mean_test_score",
    "std_test_score",
    "param_tfidf__ngram_range",
    "param_tfidf__min_df",
    "param_classifier__alpha",
]].sort_values("mean_test_score", ascending=False).reset_index(drop=True)


### O que observar

Não olhe apenas para o primeiro lugar.

Compare também:

- diferenças pequenas entre configurações;
- variação (`std_test_score`);
- custo e complexidade adicionais.

Uma configuração ligeiramente melhor, mas muito mais complexa, nem sempre é a melhor escolha de engenharia.


## 11. Avaliação final no teste preservado

Somente agora usamos o teste final.


In [ ]:
from sklearn.metrics import classification_report

best_model = search.best_estimator_
test_predictions = best_model.predict(X_test)

print(classification_report(y_test, test_predictions, digits=3))


### Regra importante

O conjunto de teste deve ser usado como avaliação final, não como instrumento iterativo de escolha.

Se continuarmos ajustando o modelo depois de olhar repetidamente para o teste, ele deixa de ser realmente independente.


## 12. Exercício guiado

Use o pipeline e os dados desta aula para testar uma grade menor:

```python
exercise_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'classifier__alpha': [0.1, 1.0],
}
```

Seu código deve:

1. criar um `GridSearchCV`;
2. usar `f1_macro` como métrica;
3. usar validação cruzada com 3 folds;
4. ajustar apenas em `X_train` e `y_train`;
5. exibir `best_score_` e `best_params_`.


In [ ]:
# Escreva sua solução aqui.

exercise_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'classifier__alpha': [0.1, 1.0],
}

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q7.hint()` e `q7.solution()`.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q7 = TILExercise(
    hint_text=(
        "Use **scikit-learn**. A classe principal é `GridSearchCV`. "
        "Passe o `pipeline` em `estimator`, `exercise_grid` em `param_grid`, `scoring='f1_macro'` e `cv=3`. "
        "Depois use `.fit(X_train, y_train)`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "from sklearn.model_selection import GridSearchCV\n\n"
        "exercise_search = GridSearchCV(\n"
        "    estimator=pipeline,\n"
        "    param_grid=exercise_grid,\n"
        "    scoring='f1_macro',\n"
        "    cv=3,\n"
        ")\n\n"
        "exercise_search.fit(X_train, y_train)\n"
        "print('Melhor score:', round(exercise_search.best_score_, 3))\n"
        "print('Melhores parâmetros:', exercise_search.best_params_)\n"
        "```"
    ),
)

print("Exercício preparado. Tente resolver antes de usar q7.hint() ou q7.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q7.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q7.solution()


## 13. Reprodutibilidade

- linguagem: Python;
- bibliotecas: `scikit-learn` e `pandas`;
- pipeline: TF-IDF + MultinomialNB;
- seleção: `GridSearchCV`;
- métrica de busca: `f1_macro`;
- validação cruzada: 3 folds;
- divisão treino/teste: `random_state=42`;
- acelerador: CPU;
- internet: desabilitada.


## 14. Resumo

Nesta aula, você aprendeu que:

- o conjunto de teste deve permanecer independente;
- validação cruzada permite comparar configurações usando apenas o treino;
- hiperparâmetros controlam o comportamento do pipeline;
- `GridSearchCV` automatiza a comparação de combinações;
- F1 macro pode ser útil quando queremos tratar classes com igual importância;
- melhor score não significa automaticamente melhor decisão de engenharia;
- tuning disciplinado reduz o risco de decisões baseadas em sorte ou vazamento de informação.

### Ideia principal

```text
Escolher um modelo é parte do experimento.
O teste final deve continuar sendo uma surpresa.
```

**Fim da Aula 7.**
